<a href="https://colab.research.google.com/github/btrailor/polly/blob/development/copy_of_sentence_transformers_rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Intro to NLP for RAG

Thanks for coming! I've tried to include as many resources as possible, but the biggest part of this notebook is of course: the code!
We'll run the main reccomended flow together, then build a RAG chatbot on a different set of data.

# Generating and Using High-Quality Embeddings in RAG

- Generate embeddings and use them to retrieve documents.
- Generate text embeddings using BGE-large-en.
- Implement FAISS for fast nearest-neighbor search.
- Re-rank to improve retrieval quality in RAG.

---

- **Embeddings** convert text into **high-dimensional numerical vectors** that capture meaning.
- They allow machines to **compare text meaning mathematically** instead of relying on exact words.
- Used in **search engines, chatbots, and RAG systems**.


- **BGE-large-en** is one of the **highest-scoring embedding models** on the [MTEB benchmark](https://huggingface.co/spaces/mteb/leaderboard).
- **Optimized for search** and **commercially viable** under Apache-2.0 license.
- **Outperforms older models** (like Universal Sentence Encoder or SBERT) in retrieval accuracy.
- **1024-dimensional embeddings** capture deeper semantic meaning, improving retrieval precision.

📌 **Alternatives:**
- [`intfloat/e5-large-v2`](https://huggingface.co/intfloat/e5-large-v2) – Another top-tier embedding model.
- [`sentence-transformers/all-mpnet-base-v2`](https://huggingface.co/sentence-transformers/all-mpnet-base-v2) – More lightweight option.

🧘☸️ **All This Will Change**
- The only thing constant in AI is change- be [ready to re-run evaluations](https://github.com/RUC-NLPIR/FlashRAG)


# Step 1: Install Dependencies

In [ ]:
!pip install sentence-transformers faiss-cpu wikipedia-api

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 68.0 MB/s eta 0:00:00
  Created wheel for wikipedia-api: filename=Wikipedia_API-0.8.1-py3-none-any.whl size=15383 sha256=c3d350938dcdc9fe2262cd9ab62daf6f9f8e1b6e4cd5e71e374aa5903f2ed465
  Stored in directory: /root/.cache/pip/wheels/33/3c/79/b36253689d838af4a0539782853ac3cc38a83a6591ad570dde
Successfully built wikipedia-api


## Step 1.5: Fetch some documents from Wikipedia

In [ ]:
import wikipediaapi

import os
import random



# Set up Wikipedia API with a custom User-Agent
wiki_wiki = wikipediaapi.Wikipedia(user_agent='MyResearchBot/1.0 (https://themultiverse.school/; liz@themultiverse.school)', language='en')

# Define categories and sample articles from different domains
categories = {
  "Computers": {
      "Computer science": ["Abstract Data Structure", "Algorithm", "Object Oriented Programming", "Scripting", "Python (programming language)"],
      "CyberSecurity": ["TCP IP", "Internet Protocol", "Computer Network", "TCP Packet", "LoRA", "OSI Model", "Layer 1", "Layer 2", "Layer 3", "Layer 4", "Layer 5", "Layer 6", "Layer 7", "Layer 8", "International Organization for Standardization", "OSHA"],
      "Machine Learning": ["Artificial intelligence", "Machine learning", "Deep learning", "Computer vision", "Natural language processing", "FAISS", "Embedding"]
  },
  "Biology": {
    "Cell biology": ["Cytoskeleton", "Cell membrane", "Endoplasmic reticulum", "Golgi apparatus", "Apoptosis"],
    "Genetics": ["Epigenetics", "Gene expression", "CRISPR", "DNA replication", "Genetic drift"],
    "Food Web": ["Trophic cascade", "Keystone species", "Ecological pyramid", "Energy flow (ecology)", "Biogeochemical cycle"],
    "Microbiology": ["Bacteriophage", "Gram-positive bacteria", "Archaea", "Biofilm", "Extremophiles"],
    "Human anatomy": ["Circulatory system", "Endocrine system", "Nervous system", "Musculoskeletal system", "Digestive system"],
    "Mitochondria": ["ATP synthase", "Mitochondrial DNA", "Oxidative phosphorylation", "Mitochondrial diseases", "Endosymbiotic theory"],
    "Phylogenetics": ["Cladistics", "Evolutionary tree", "Molecular phylogenetics", "Common descent", "Homology (biology)"]
  },
  "Chemistry": {
    "Organic chemistry": ["Functional groups", "Alkene", "Aromaticity", "Polymerization", "Carbohydrates"],
    "Inorganic chemistry": ["Coordination complex", "Transition metal", "Crystallography", "Lanthanides", "Actinides"],
    "Analytical chemistry": ["Chromatography", "Spectroscopy", "Mass spectrometry", "Electrochemical analysis", "Nuclear magnetic resonance"],
    "Physical chemistry": ["Quantum chemistry", "Thermodynamics", "Statistical mechanics", "Chemical kinetics", "Molecular dynamics"],
    "Biochemistry": ["Enzyme kinetics", "Protein folding", "Lipid metabolism", "Glycolysis", "Signal transduction"]
  },
  "Geology": {
    "Plate tectonics": ["Subduction zone", "Mid-ocean ridge", "Continental drift", "Transform fault", "Rift valley"],
    "Mineralogy": ["Silicate minerals", "Feldspar", "Quartz", "Mohs scale of mineral hardness", "Crystal habit"],
    "Volcano": ["Stratovolcano", "Shield volcano", "Pyroclastic flow", "Volcanic explosivity index", "Supervolcano"],
    "Earthquake": ["Seismic wave", "Richter scale", "Fault mechanics", "Liquefaction", "Tsunami"],
    "Geological history of Earth": ["Hadean eon", "Cambrian explosion", "Snowball Earth", "K-Pg extinction event", "Great Oxygenation Event"],
    "Igneous Rock": ["Basalt", "Granite", "Magma differentiation", "Intrusive rock", "Plutonic rock"]
  },
  "History": {
    "World War II": ["Battle of Stalingrad", "Manhattan Project", "D-Day", "Holocaust", "Blitzkrieg"],
    "Ancient Egypt": ["Pharaoh", "Hieroglyphics", "Valley of the Kings", "Mummification", "Great Pyramid of Giza"],
    "Renaissance": ["Humanism (Renaissance)", "Leonardo da Vinci", "Medici family", "Florence during the Renaissance", "Printing press"],
    "Industrial Revolution": ["Steam engine", "Factory system", "Textile industry", "Urbanization", "Luddites"],
    "Cold War": ["Cuban Missile Crisis", "Space Race", "Berlin Wall", "McCarthyism", "NATO"]
  },
  "Art": {
    "Impressionism": ["Claude Monet", "Edgar Degas", "Pierre-Auguste Renoir", "Plein air painting", "Color theory"],
    "Cubism": ["Pablo Picasso", "Georges Braque", "Analytic Cubism", "Synthetic Cubism", "Still Life with Chair Caning"],
    "Renaissance art": ["Michelangelo", "Sistine Chapel ceiling", "Raphael", "Leonardo da Vinci’s notebooks", "Linear perspective"],
    "Sculpture": ["Rodin", "Bronze casting", "Marble sculpture", "Gothic sculpture", "Greek classical sculpture"],
    "Abstract art": ["Wassily Kandinsky", "Color field painting", "Abstract expressionism", "Suprematism", "De Stijl"],
    "Dadaism": ["Marcel Duchamp", "Readymades", "Cabaret Voltaire", "Tristan Tzara", "Anti-art movement"],
    "Absurdism": ["Albert Camus", "The Myth of Sisyphus", "Theatre of the Absurd", "Samuel Beckett", "Waiting for Godot"]
  }
}

# Create the cache directory
cache_dir = "data/wikipedia_cache"
os.makedirs(cache_dir, exist_ok=True)

# Function to save Wikipedia text to cache
def save_to_cache(title, text):
    filename = os.path.join(cache_dir, f"{title.replace(' ', '_')}.txt")
    with open(filename, "w", encoding="utf-8") as file:
        file.write(text)

# Function to load Wikipedia text from cache
def load_from_cache(title):
    filename = os.path.join(cache_dir, f"{title.replace(' ', '_')}.txt")
    if os.path.exists(filename):
        with open(filename, "r", encoding="utf-8") as file:
            return file.read()
    return None

# Function to fetch Wikipedia page content with caching
def get_wikipedia_text(title):
    # Check if the page is already cached
    cached_text = load_from_cache(title)
    if cached_text:
        return cached_text

    page = wiki_wiki.page(title)

    # If it's a disambiguation page, follow the first few linked pages
    if page.exists():
        if "may refer to:" in page.text[:200]:  # Check if it's a disambiguation page
            linked_pages = list(page.links.keys())[:5]  # Grab first few related links
            for linked_title in linked_pages:
                sub_page = wiki_wiki.page(linked_title)
                if sub_page.exists() and len(sub_page.text) > 500:  # Ensure meaningful content
                    save_to_cache(linked_title, sub_page.text[:2000])  # Save to cache
                    return sub_page.text[:2000]

        # Save the fetched page to cache and return it
        save_to_cache(title, page.text[:2000])
        return page.text[:2000]

    return None

# Function to get additional Wikipedia pages from related categories
def get_category_pages(category_name, max_pages=5):
    category_page = wiki_wiki.page(f"Category:{category_name}")
    pages = []

    if category_page.exists():
        for title, page in category_page.categorymembers.items():
            if page.ns == 0:  # Only fetch articles (not subcategories)
                text = get_wikipedia_text(title)
                if text:
                    pages.append(text)
                if len(pages) >= max_pages:
                    break
    return pages

# Fetch and store documents
documents = []
document_name_index = {}
document_page_index = {}

for category, topics in categories.items():
    for topic in topics:
        print("Getting ", topic)
        for page in topics[topic]:
            text = get_wikipedia_text(page)
            if text:
                document_name_index[page.lower()] = text
                document_page_index[text[:50]] = page.lower()
                documents.append(text)

    # Also pull a few pages from the Wikipedia category
    documents.extend(get_category_pages(category, max_pages=5))


print(f"Collected {len(documents)} Wikipedia documents. Cached files saved in: {cache_dir}")





Getting  Computer science
Getting  CyberSecurity
Getting  Machine Learning
Getting  Cell biology
Getting  Genetics
Getting  Food Web
Getting  Microbiology
Getting  Human anatomy
Getting  Mitochondria
Getting  Phylogenetics
Getting  Organic chemistry
Getting  Inorganic chemistry
Getting  Analytical chemistry
Getting  Physical chemistry
Getting  Biochemistry
Getting  Plate tectonics
Getting  Mineralogy
Getting  Volcano
Getting  Earthquake
Getting  Geological history of Earth
Getting  Igneous Rock
Getting  World War II
Getting  Ancient Egypt
Getting  Renaissance
Getting  Industrial Revolution
Getting  Cold War
Getting  Impressionism
Getting  Cubism
Getting  Renaissance art
Getting  Sculpture
Getting  Abstract art
Getting  Dadaism
Getting  Absurdism
Collected 200 Wikipedia documents. Cached files saved in: data/wikipedia_cache


## Step 2: Load the Qwen3 Embedding Model and Encode Documents


In [ ]:
from sentence_transformers import SentenceTransformer
# Load the Qwen3 model for generating embeddings
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
# Generate embeddings
document_embeddings = model.encode(documents, convert_to_numpy=True)

# Print embedding shape
print("Embedding Shape:", document_embeddings.shape)  # Expect (>=100, 1024)

## Step 3: Using FAISS for Efficient Search

Now, we store the embeddings in a FAISS index to enable fast similarity search.

FAISS (Facebook AI Similarity Search) is a library for fast nearest neighbor search in high-dimensional spaces. It optimizes both exact and approximate search.

How It Works:

- Flat Index (IndexFlatL2): Exhaustive, computes exact distances.
- Quantization & Graph-Based Methods: Reduces memory and speeds up searches (e.g., IVF, HNSW).
- GPU Acceleration: Supports efficient large-scale indexing.

Advantages:

- Scalable: Handles millions of vectors.
- Fast: Optimized for both exact and approximate search.
- Flexible: Multiple indexing strategies balance speed, accuracy, and memory use.

When to Use FAISS:

- Large-scale similarity search
- Need for GPU acceleration
- Trade-offs between speed, memory, and accuracy are required

In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.8 MB/s eta 0:00:00


In [ ]:
query = "what did ww2 do to art"

In [ ]:
# Set up groq client
from google.colab import userdata
groq_key = userdata.get('GROQ_API_KEY')
import json

from groq import Groq

client = Groq(api_key=groq_key)

def prompt_groq(prompt):
  print(prompt)
  chat_completion = client.chat.completions.create(
      messages=[
          {
              "role": "user",
              "content": prompt,
          }
      ],
      model="qwen/qwen3-32b",
  )

  return chat_completion.choices[0].message.content


def prompt_groq_with_tool_and_force_use(prompt,tools):
  chat_completion = client.chat.completions.create(
      messages=[
          {
              "role": "user",
              "content": prompt,
          }
      ],
      model="qwen/qwen3-32b",
      tools=tools,
      tool_choice=tools[0]
  )
  print(chat_completion)
  tool_calls = [json.loads(call.function.arguments) for call in chat_completion.choices[0].message.tool_calls]
  return tool_calls


In [ ]:
query_expansion_prompt = f"The user has made the following query: {query}. Please rewrite and expand on the query to improve the search results retrieved. DO NOT answer the query, only rewrite the query so that we're able to search our document index more effectively."
expanded_query = prompt_groq(query_expansion_prompt)
print(expanded_query)
original_query = query
query = expanded_query
query = query.split("</think>")[-1]
print(query)

In [ ]:
import faiss

# Initialize FAISS index
embedding_size = document_embeddings.shape[1]  # 1024
index = faiss.IndexHNSWFlat(embedding_size,5)

# Add embeddings to FAISS index
index.add(document_embeddings)

print("FAISS index created with", index.ntotal, "documents.")

# Save FAISS-HNSW Index
faiss.write_index(index, "faiss_index.bin")


print("Saved FAISS index to faiss_index.bin")

#### 📌 Alternative: TensorFlow Approximate Nearest Neighbors (ANN)
You might have a reason to use other methods! Here's a handy table:

| Feature            | **FAISS** (Facebook AI Similarity Search) | **General ANN Methods** (e.g., HNSW, BallTree, KDTree) |
|-------------------|----------------------------------|--------------------------------|
| **Speed**        | Highly optimized for fast search, supports GPU acceleration | Varies by method; HNSW is fast, but KDTree struggles in high dimensions |
| **Scalability**  | Handles millions to billions of vectors efficiently | Some ANN methods scale well (HNSW), others (KDTree, BallTree) do not |
| **Accuracy**     | Configurable for exact or approximate search | Varies; HNSW offers high accuracy, others may sacrifice accuracy for speed |
| **Memory Usage** | Optimized via quantization and compression | Varies; HNSW uses more memory, others may be more efficient |
| **Implementation Complexity** | Easy-to-use library with multiple indexing strategies | Some methods require deeper parameter tuning and domain expertise |
| **Parallelization** | Supports multi-threading and GPU acceleration | Many ANN methods lack GPU support |
| **Use Case Fit** | Best for large-scale, high-dimensional similarity search | Some ANN methods (KDTree, BallTree) perform better in low dimensions |


## 3. Using Embeddings for Retrieval

Now, let's query the knowledge base using FAISS and find the top 5 most relevant documents

In [ ]:
# Function to retrieve top-k documents
def retrieve_top_k_documents(query, top_k=3):
    query_embedding = model.encode([query], convert_to_numpy=True)
    _, indices = index.search(query_embedding, top_k)
    return [documents[i] for i in indices[0]]

# Ask user for a question

retrieved_docs = retrieve_top_k_documents(query, top_k=10)

# Display results
print("\nTop Retrieved Documents:")
for idx, doc in enumerate(retrieved_docs, 1):
    print(f"{idx}. {doc}")

## 5. Re-Ranking: Use a Neural Re-Ranker to Improve Retrieval Precision

Problem: The retrieved documents may not always be perfectly ranked.
Solution: Use a cross-encoder reranker (e.g., `cross-encoder/ms-marco-MiniLM-L-12-v2`).

In [ ]:
from sentence_transformers import CrossEncoder

# Load a neural re-ranking model
reranker = CrossEncoder("Qwen/Qwen3-Reranker-0.6B")

# Add a padding token to the model
if reranker.model.config.pad_token_id is None:
    reranker.model.config.pad_token_id = reranker.model.config.eos_token_id


# Function to re-rank top-k documents
def rerank_documents(query, retrieved_docs):
    # Create query-document pairs for scoring
    pairs = [[query, doc] for doc in retrieved_docs]

    # Get relevance scores
    scores = reranker.predict(pairs)

    # Sort documents by score
    ranked_docs = [doc for _, doc in sorted(zip(scores, retrieved_docs), reverse=True)]

    return ranked_docs

# Retrieve and re-rank top documents
retrieved_docs = retrieve_top_k_documents(query,10)
reranked_docs = rerank_documents(query, retrieved_docs)

print("\nTop Re-Ranked Documents:")
for idx, doc in enumerate(reranked_docs, 1):
    print(f"{idx}. {doc}")

### Hierarchical Navigable Small Worlds (HNSW) Indexing
Hierarchical Navigable Small Worlds (HNSW) Indexing

What is HNSW?

HNSW (Hierarchical Navigable Small Worlds) is an advanced approximate nearest neighbor (ANN) algorithm that speeds up retrieval while maintaining high accuracy. Unlike FAISS IndexFlatL2, it builds a multi-layered graph for efficient navigation.

How It Works
- Hierarchical Structure: Vectors are arranged in layers; the top has fewer nodes, lower layers have more.
- Graph-Based Navigation: Each point connects to neighbors, forming a graph. Searches:
- Start at a random top-layer entry
- Navigate toward the query
- Descend layers for precision
- Small World Properties: Most nodes are reachable in a few steps.

Advantages Over Flat Indexing
- **Scalability**: Handles millions of vectors with sub-linear search time; flat indices are linear.
- **Speed-Accuracy Trade-off**: Tunable via `M` and `ef_construction`.
- **Memory Efficiency**: More efficient than a full linear scan despite higher memory use.
- **Real-World Performance**: Outperforms many ANN algorithms when properly tuned.

When to Use HNSW
- Dataset >10,000 documents
- Query speed is critical
- High recall accuracy (>95%) needed without a full scan
- Sufficient memory available for graph storage

In [ ]:
import faiss

# Create HNSW index
embedding_size = document_embeddings.shape[1]  # 1024 for BGE-large-en
M = 16  # Number of connections per layer (higher = more accurate but slower to build)
ef_construction = 200  # Controls index quality (higher = better recall but slower to build)

# Create the index
index = faiss.IndexHNSWFlat(embedding_size, M)
index.hnsw.efConstruction = ef_construction
index.hnsw.efSearch = 128  # Controls search accuracy/speed trade-off

# Add vectors to the index
index.add(document_embeddings)

faiss.write_index(index, "hnsw_index.bin")
print("Saved HNSW index to hnsw_index.bin")

# Exercise: Implement a Simple RAG-Based Q&A System, then compare models

Exercise Objectives

- Use wikipedia documents as the basis for your RAG system.
-	Use BGE embeddings to index wikipedia documents
-	Accept queries from the user
- Pass the queries to [Deepseek's R1-Distillation of Qwen](https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-7B) for query expansion
- Retrieve the best documents
-	Use a Neural Reranker to re-rank results retrieved
- Extend FIASS with HNSW
- Use [Deepseek's R1-Distillation of Qwen](https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-7B) to interpret the retrieved documents into an answer to your question (or an internally-hosted LLM of your choice)

In [ ]:
!pip install groq

In [ ]:
## here's where your code goes
## given the context and the user query
## generate a response using a language model
from google.colab import userdata
groq_key = userdata.get('GROQ_API_KEY')

from groq import Groq

client = Groq(api_key=groq_key)

prompt = "Using the context above, and the rewritten user's query, answer the query, do not rewrite the query. Don't use any information outside of the articles you've been given, don't hallucinate or make anything up, or use your own memory to answer the question. Only include material from the articles in your answer."
context = "\n".join(reranked_docs)
full_prompt = f"{context}  {prompt}  {query}"

def prompt_groq(prompt):
  print(prompt)
  chat_completion = client.chat.completions.create(
      messages=[
          {
              "role": "user",
              "content": prompt,
          }
      ],
      model="qwen/qwen3-32b",
  )

  return chat_completion.choices[0].message.content

result = prompt_groq(full_prompt)
print(result)

In [ ]:
import json
results = result.split("</think>")[-1]
print(results)

prompt = """
An AI model was given the contex above, and a rewritten query to produce a final result according to the rewritten query.
Please review the documents, and indicate which documents were the most helpful in answering the user's query by mentioning their names.
Use the reward_articles tool to specify which articles were most relevant to the query to reward the reranking model.
"""

doc_titles = [document_page_index.get(doc[:50]) for doc in reranked_docs if document_page_index.get(doc[:50])]
docs_with_titles = [f"## {title}:\n {doc[:200]}" for title, doc in zip(doc_titles, reranked_docs) if document_page_index.get(doc[:50])]

# context = "\nNew Article:".join([doc[0:500] for doc in reranked_docs])
context = "\n".join(docs_with_titles)
full_prompt = f"Articles:{context}\nRewritten Query: {query}\n Results: {results}\n {prompt}"

relevance_results = prompt_groq_with_tool_and_force_use(full_prompt, [
  {
    "type": "function",
    "function": {
      "name": "reward_articles",
      "description": "Specify which articles were most relevant to the query to reward the reranking model",
      "parameters": {
        "type": "object",
        "properties": {
          "relevant_articles": {
            "type": "array",
            "description": "A Python-Formatted List of article titles that were the most helpful, eg ['article 1', 'article 2', 'article 3']. Please return multiple articles if multiple articles were useful. Return ALL related articles that were used in the final answer. This will be loaded with json.loads()."
          }
        },
        "required": ["relevant_articles"]
      }
    }
  }
])

print(relevance_results)
relevant_articles = []
try:
  for relevance_result in relevance_results:
    relevant_articles.append(relevance_result.get("relevant_articles"))
except Exception as e:
  print(e)
  relevant_articles = []
finally:
  print(relevant_articles)